# Oppitunti 18: AI-agenttien turvaaminen kryptografisilla kuiteilla

## Käytännön harjoitus

Tämä harjoitus ohjaa neljän tehtävän läpi:

1. **Allekirjoita ensimmäinen kuitisi** agenttityökalu-kutsulle ja vahvista se.
2. **Muuta kuittia** ja katso, kun vahvistus epäonnistuu.
3. **Rakenna kolmen kuitin ketju** ja varmista ketjun eheys.
4. **Kääri Microsoft Agent Framework -työkalu kutsu** siten, että jokainen toiminto tuottaa kuitin.

Kaikki kryptografiset primitiivit ovat tuotu hyvin ylläpidetyistä kirjastoista (`pynacl` Ed25519:lle, `jcs` RFC 8785 kannoniselle JSONille, `hashlib` Pythonin standardikirjastosta SHA-256:lle). Kuitin käsittelylogiikka itsessään on tavallista Pythonia, jota voit lukea ja muokata.

Suorita solut järjestyksessä. Jokainen osio on lyhyt ja itsenäinen.


## Asennus

Asenna molemmat riippuvuudet. Molemmilla on sallivat lisenssit (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Aputyökalut

Nämä kaksi apuria käsittelevät base64url-koodauksen (ilman täyteosia) ja mielivaltaisten objektien SHA-256-tiivistämisen. Ne pitävät muun muistiinpanon keskittyneenä kuitin logiikkaan itseensä.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Osa 1: Allekirjoita ensimmäinen kuittisi

Kuvittele, että **Contoso Travel** -yrityksen agenttimme on juuri etsinyt lentoja Sydneystä Los Angelesiin asiakkaalle. Haluamme tallentaa tämän työkalupuhelun allekirjoitettuna kuittina, jotta tuleva tarkastaja voi vahvistaa sen luottamatta meihin.

### Vaihe 1.1: Luo allekirjoitusavain

Tuotantoympäristössä agentin allekirjoitusavain sijaitsisi laitteistoturvamoduulissa (HSM), Azure Key Vaultissa tai vastaavassa suojatussa säilytystilassa. Tätä oppituntia varten luomme uuden avaimen muistiin.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Vaihe 1.2: Rakenna kuittipayload

Payload sisältää kaiken, mitä haluamme kuitin vahvistavan: kuka toimi, mitä työkalua, millä argumenteilla, mitä tuli takaisin, minkä politiikan alaisena ja milloin. Tiivistämme argumentit ja tuloksen emmekä sisällytä niitä suoraan, jotta kuitti ei vuoda arkaluontoista sisältöä.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Vaihe 1.3: Allekirjoita ja kokoaa kuitti

Kolme vaihetta:

1. Normalisoi sisältö JCS:llä, jotta kaksi toteutusta, jotka tuottavat saman loogisen kuitin, tuottavat tavuidenttiset tavut.
2. Hashaa normalisoidut tavut SHA-256:lla.
3. Allekirjoita hash Ed25519:n yksityisavaimella.

Allekirjoitus liitetään sitten alkuperäiseen sisältöön lopullisen kuitin tuottamiseksi.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Vaihe 1.4: Tarkista kuitti

Tarkistus kääntää prosessin päinvastaiseksi. Poistamme allekirjoituksen, laskemme kanonisen tiivisteen uudelleen ja tarkistamme allekirjoituksen kuittiin sisältyvää julkista avainta vastaan.

Tarkastajan, joka suorittaa tämän varmennuksen, ei tarvitse meiltä muuta kuin itse kuitti. Ei palvelukutsua, ei avainhakemistoa tai luottamusta.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Sinun pitäisi nähdä `Kuitti on voimassa: Tosi`. Agentti on tuottanut ensimmäisen kryptografisesti allekirjoitetun auditointitietueensa.


## Osa 2: Muuntele kuittia

Kuitin koko pointti on, että sen muuttelu näkyy. Todistetaan se.

Muokkaamme kuittia täsmälleen yhdellä merkillä ja katsomme, miten varmennus epäonnistuu.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Mitä juuri tapahtui?

Kun muutimme `policy_id`-arvoa, kanoniset tavut muuttuivat. Näiden tavujen SHA-256 tiiviste muuttui. Allekirjoitus (joka oli alkuperäisen tiivisteen päälle) ei enää vastaa uutta tiivistettä. Varmistus palauttaa oikein `False`.

Ei ole mahdollista muuttaa mitä tahansa kuittikenttää niin, että sen voi edelleen varmistaa, ellei hyökkääjällä ole yksityistä avainta. Niin kauan kuin yksityinen avain on avainvarastossa ja julkinen avain on julkaistu, väärentämistä ei voi peittää.

Kokeile itse: muuta ylläolevassa solussa `tool_name`-, `agent_id`- tai `timestamp`-arvoa ja suorita uudelleen. Jokainen muutos tuottaa virheellisen kuitin.


## Osa 3: Kytke kuitit ketjuun

Yksi kuitti suojaa yhtä toimintoa. Useimmat toimijat suorittavat useita toimintoja. Koko sekvenssin tekemiseksi vääristymättömäksi kytkemme jokaisen kuitin edelliseen sisällyttämällä edellisen kuitin tiivisteen uuden kuitin tietosisältöön.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Jos kukaan poistaa tai muuttaa kuitin järjestystä, ketju katkeaa juuri siinä kohdassa. Minkä tahansa myöhemmän kuitin vahvistus epäonnistuu, koska sen `previous_receipt_hash` ei enää vastaa edeltäjänsä todellista tiivistettä.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Katkaise nyt ketju muokkaamalla keskimmäistä kuittia ja tarkista uudelleen. Muokattu kuitti epäonnistuu allekirjoituksen tarkistuksessa, JA seuraava kuitti epäonnistuu ketjuyhteen tarkistuksessa (koska sen `previous_receipt_hash` ei enää vastaa muokatun keskimmäisen kuitin hajautetta).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Kuitti 0 vahvistuu edelleen (sitä ei muutettu eikä sillä ole riippuvaista edeltäjää). Kuitti 1 epäonnistuu allekirjoituksen tarkistuksessa, koska muutimme `tool_args_hash`-arvoa. Kuitti 2 epäonnistuu ketjulinkki-tarkistuksessa, koska sen `previous_receipt_hash` laskettiin alkuperäistä (nyt muutettua) kuittia 1 vastaan.

Vaikka hyökkääjä allekirjoittaisi uudelleen muutetun kuitin 1 (mikä ei onnistu ilman yksityisavainta), ketjulinkkien epäyhtenäisyys kuitissa 2 paljastaisi silti manipuloinnin. Muutoksen piilottamiseksi hyökkääjän pitäisi allekirjoittaa uudelleen jokainen kuitti muutoskohdasta eteenpäin, mikä edellyttää yksityisavaimen hallussapitoa.


## Osa 4: Kääri agenttityökalukutsu kuittauksen allekirjoituksella

Todellisessa käyttöönotossa et halua, että jokaisen agentin kirjoittajan täytyy muistaa kutsua `make_receipt`. Haluat, että kuittauksen allekirjoitus on automaattinen jokaista työkalun kutsua varten.

Tässä on yksinkertaisin malli: kääreluokka, joka ottaa vastaan minkä tahansa kutsuttavan työkalufunktion ja palauttaa version siitä, joka lähettää kuitin. Tämä mukautuu mihin tahansa agenttikehykseen, mukaan lukien Microsoft Agent Framework (`agent_framework.foundry`).

Jos sinulla ei ole Microsoft Foundry -projektia käytössä, alla oleva paikallinen mallinnus demonstroi tätä mallia silti.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### Integrointi Microsoft Agent Frameworkin kanssa

Yllä oleva `ReceiptedTool`-kääre on kehysriippumaton. Jos haluat käyttää sitä Microsoft Agent Frameworkilla rakennetun agentin sisällä, rekisteröi kääritty funktio työkaluna. Luonnos (voisit korvata mallin todellisella Microsoft Foundry -työkalun rekisteröinnillä):

```python
# Pseudokoodi, joka näyttää integroinnin muodon.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="Olet Contoso Travelin agentti ...",
#     tools=[receipted_lookup],   # kääritty työkalu, ei raakafunktio
# )
# response = agent.run("Etsi lennot Sydneystä Los Angelesiin kesäkuussa.")
#
# # Suorituksen jälkeen jokaisella agentin tekemällä työkalukutsulla on allekirjoitettu kuitti:
# audit_chain = receipted_lookup.receipts
```

Agenttikehyksen ei tarvitse tietää mitään kuiteista. Kuitin allekirjoitus on kääritty työkalun ympärille, ei kiinnitetty kehykseen. Näin lisäät jäljitettävyyden olemassa olevaan agenttikoodiin ilman agentin uudelleenkirjoittamista.


## Kertaus ja haastava lisätehtävä

Olet tehnyt:

- Luonut Ed25519-avainparin.
- Rakentanut ja allekirjoittanut kuitin agenttityökalukutsulle.
- Vahvistanut kuitin offline-tilassa käyttäen vain julkista avainta.
- Muokannut kuittia ja havainnut vahvistuksen epäonnistumisen.
- Rakentanut hajautettu ketju kolmen kuitin sarjalle.
- Muokannut ketjun keskikohtaa ja havainnut sekä allekirjoituksen epäonnistumisen että ketjuyhdistevirheen.
- Kietonut työkalufunktion automaattiseen kuitin allekirjoittamiseen.

**Haastava lisätehtävä.** Laajenna kuittiskaalaa lisäämällä `request_id`-kenttä (UUID hajautettuun jäljitykseen). Päivitä `make_receipt` ottamaan tämä kenttä mukaan ja varmista, että kuitit edelleen vahvistuvat päästä päähän. Muokkaa kenttää allekirjoittamisen jälkeen ja varmista, että vahvistus epäonnistuu. Tämä pakottaa ymmärtämään, miten jokainen tavukoodi kanonisen koodauksen sisällä vaikuttaa allekirjoitukseen.

**Tärkeä raja.** Kuitit todistavat kolme asiaa, eikä muuta: attribuution (tämä avain allekirjoitti tämän sisällön), eheyden (sisältö ei ole muuttunut allekirjoituksen jälkeen) ja järjestyksen (tämä kuitti tuli kyseisen kuitin jälkeen). Ne eivät todista, että agentin toimi oli oikein, että `policy_id`-niminen politiikka todella arvioitiin, tai että agentti noudatti kaikkia sääntöjä. Kuitit ovat perusta. Hallinto on järjestelmä, jonka rakennat tämän päälle.

Lue oppitunnin README uudelleen tämän rajan huomioiden. Usein tiimit tekevät sen virheen kuiteista, että "meillä on kuitit" tarkoittaa "meitä hallinnoidaan". Näin ei ole. Kuitit tekevät agentin toiminnan tarkastettavaksi. Ne eivät tee sitä oikeaksi.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
